In [ ]:
%jsroot on

import ROOT
import ipywidgets as widgets
from ipywidgets import interact, interact_manual
from IPython.display import display

import numpy as np
 
# for illustration, a sinusoidal function with six parameters
myFunction = ROOT.TF1("myExampleFunction", "[0] + [1] * x + [2]*sin([3] * x) + [4]*cos([5] * x)", -5, 5)
 
# declare the parameters as a numpy.array or array.array 
#myParams = np.array([0.04, 0.4, 3.0, 3.14, 1.5, 3.14])

c = ROOT.TCanvas()
f1 = ROOT.TF1("func1", "sin(x)", 0, 10)
display(w)
w.value = 3.0
er = 3.0
myParams = np.array([er] * 6) 
myFunction.SetParameters(myParams)
myFunction.Draw()
c.Draw() # Necessary to make the graphics show!

In [ ]:
%jsroot on

import ROOT
import ipywidgets as widgets

# TF1
f = ROOT.TF1(
    "f",
    "[0] + [1]*x + [2]*sin([3]*x) + [4]*cos([5]*x)",
    -5, 5
)

def update(p0):
    c = ROOT.TCanvas("c", "c", 600, 400)

    f.SetParameters(p0, p0, p0, p0, p0, p0)
    f.Draw()
    c.Draw()
    c.Update()
    return c   # ← 关键：必须 return

slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=5.0,
    step=0.1,
    description="p"
)

widgets.interact(update, p0=slider)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

x = np.linspace(-5, 5, 400)

def f(x, p0, p1, p2, p3, p4, p5):
    return (
        p0
        + p1 * x
        + p2 * np.sin(p3 * x)
        + p4 * np.cos(p5 * x)
    )

def update(p):
    clear_output(wait=True)

    y = f(x, p, p, p, p, p, p)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(x, y)
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.set_title(f"p = {p:.2f}")
    ax.grid(True)

    plt.show()

slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=5.0,
    step=0.1,
    description="p"
)

#display(slider)
widgets.interact(update, p=slider)


interactive(children=(FloatSlider(value=1.0, description='p', max=5.0), Output()), _dom_classes=('widget-inter…

<function __main__.update(p)>

In [2]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output

file_path = "../rapidityTree/tree_Au197pnHFB14Au197pnHFB14_62p4_alpha5.00_cent0_5.root"

with uproot.open(file_path) as f:
    tree = f["t"]
    arrays = tree.arrays(
        ["evt", "isProton", "y_final"],
        library="np"
    )

evt = arrays["evt"]
isProton = arrays["isProton"]
y_final = arrays["y_final"]


In [3]:
def proton_multiplicity(y_cut):
    # 选择 proton 且 |y| < y_cut
    mask = (isProton == 1) & (np.abs(y_final) < y_cut)

    evt_selected = evt[mask]

    # event-by-event 计数
    unique_evt, counts = np.unique(evt_selected, return_counts=True)

    return counts


In [7]:
def update(y_cut):
    clear_output(wait=True)

    mult = proton_multiplicity(y_cut)
    N = mult.astype(float)

    # cumulants
    c1 = N.mean()
    c2 = np.mean((N - c1)**2)
    c3 = np.mean((N - c1)**3)
    c4 = np.mean((N - c1)**4) - 3 * c2**2

    fig, ax = plt.subplots(figsize=(7, 4))

    # multiplicity histogram
    bins = np.arange(N.max() + 2) - 0.5
    ax.hist(N, bins=bins)

    ax.set_xlabel("Proton multiplicity")
    ax.set_ylabel("Events")
    ax.set_title(f"|y| < {y_cut:.2f}")
    ax.grid(True)

    # text box on the right
    text = (
        f"N = {len(N)}\n"
        f"c1 = {c1:.4f}\n"
        f"c2 = {c2:.4f}\n"
        f"c3 = {c3:.4f}\n"
        f"c4 = {c4:.4f}"
    )

    ax.text(
        1.02, 0.95,
        text,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    plt.tight_layout()
    plt.show()


slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=3.0,
    step=0.1,
    description="y_cut",
    continuous_update=True
)


widgets.interact(update, y_cut=slider)

interactive(children=(FloatSlider(value=0.5, description='y_cut', max=3.0), Output()), _dom_classes=('widget-i…

<function __main__.update(y_cut)>